# Densest Subgraph

## Learning Objectives

1. **Define** subgraph density and the densest subgraph problem
2. **Implement** the greedy 2-approximation algorithm
3. **Explain** why the greedy achieves a 2-approximation
4. **Describe** Goldberg's exact algorithm and its complexity

## Subgraph Density

For a subgraph induced by vertex set $S$:

$$\rho(S) = \frac{|E(S)|}{|S|}$$

where $|E(S)|$ = edges with both endpoints in $S$.

**Densest subgraph problem:** find $S^*$ maximizing $\rho(S)$.

**Applications:**
- Finding dense communities (spam rings, bot networks)
- Graph mining: finding clique-like structures
- Bioinformatics: dense gene co-expression modules
- Link spam detection in web graphs

In [ ]:
from collections import defaultdict

def density(adj, S):
    S = set(S)
    edges = sum(1 for u in S for v in adj[u] if v in S) // 2
    return edges / len(S) if S else 0.0

# Build a graph with a planted dense subgraph
adj = defaultdict(set)
# Dense core: nodes 0-5 (clique)
for i in range(6):
    for j in range(i+1,6):
        adj[i].add(j); adj[j].add(i)
# Sparse periphery: nodes 6-15
import random; rng = random.Random(42)
for i in range(6,16):
    for j in range(6,16):
        if i < j and rng.random() < 0.1:
            adj[i].add(j); adj[j].add(i)
# Some connections core-periphery
for i in range(6):
    v = rng.randint(6,15)
    adj[i].add(v); adj[v].add(i)

nodes = list(adj.keys())
print(f"Graph: {len(nodes)} nodes, {sum(len(v) for v in adj.values())//2} edges")
print(f"Density of clique {list(range(6))}: {density(adj, range(6)):.3f}")
print(f"Density of all nodes: {density(adj, nodes):.3f}")

## Greedy 2-Approximation

**Algorithm (Charikar 2000):**
1. Let $S = V$
2. Repeat until $|S| = 1$:
   - Remove the minimum-degree node from $S$
3. Return the $S$ with maximum density seen at any step

**Why 2-approximation:**

At the step when we remove node $v$ with degree $d_v$ in $G[S]$:
- $d_v \leq$ average degree in $G[S] = 2\rho(S)$
- So $d_v \leq 2\rho(S^*)$ (since $\rho(S) \leq \rho(S^*)$ for any $S$)
- Removing $v$ costs at most $d_v / |S|$ in density
- Therefore the best $S$ seen achieves at least $\rho(S^*) / 2$

**Complexity:** $O(m)$ with a priority queue.

In [ ]:
import heapq

def greedy_densest_subgraph(adj):
    """Greedy 2-approximation for densest subgraph. Returns (best_density, best_S)."""
    deg = {u: len(adj[u]) for u in adj}
    current = set(adj.keys())
    # Min-heap of (degree, node)
    heap = [(d, u) for u, d in deg.items()]
    heapq.heapify(heap)
    removed = set()

    best_rho = density(adj, current)
    best_S   = set(current)

    while len(current) > 1:
        # Find min-degree node still in current
        while heap and (heap[0][1] in removed or heap[0][0] != deg[heap[0][1]]):
            heapq.heappop(heap)
        if not heap: break
        _, v = heapq.heappop(heap)
        # Update neighbors' degrees
        for u in adj[v]:
            if u in current:
                deg[u] -= 1
                heapq.heappush(heap, (deg[u], u))
        current.remove(v); removed.add(v)
        rho = density(adj, current)
        if rho > best_rho:
            best_rho = rho; best_S = set(current)

    return best_rho, best_S

best_rho, best_S = greedy_densest_subgraph(adj)
print(f"Greedy densest subgraph: density={best_rho:.3f}, nodes={sorted(best_S)}")
print(f"True clique density: {density(adj, range(6)):.3f}")
print(f"Approximation ratio: {density(adj, range(6))/best_rho:.3f} (should be ≤ 2)")

## Goldberg's Exact Algorithm

Goldberg (1984) showed the densest subgraph can be found exactly via a sequence of max-flow computations:

**Binary search on $\rho$:** For a target density $g$, build a flow network and check if a subgraph with density > $g$ exists.

**Flow network construction:**
- Source $s$, sink $t$
- For each edge $(u,v)$: add edges $s→u, s→v$ with capacity 1; add edges $u→t, v→t$ with capacity $g$; add edges $u↔v$ with capacity $\infty$
- A min-cut in this network reveals the densest subgraph

**Complexity:** $O(n^3 \log n)$ — too slow for large graphs.

**Practical choice:** use greedy for large graphs; Goldberg for small exact computation.

In [ ]:
# Verify greedy approximation on random graphs
import random, math

def brute_force_densest(adj):
    """Exact densest subgraph by brute force. O(2^n) — only for tiny graphs."""
    from itertools import combinations
    nodes = list(adj.keys())
    best = (0, [])
    for r in range(2, len(nodes)+1):
        for S in combinations(nodes, r):
            rho = density(adj, S)
            if rho > best[0]: best = (rho, list(S))
    return best

rng2 = random.Random(99)
print(f"{'n':>4} {'exact':>8} {'greedy':>8} {'ratio':>8}")
for n in [6, 8, 10, 12]:
    g = defaultdict(set)
    for i in range(n):
        for j in range(i+1,n):
            if rng2.random() < 0.4:
                g[i].add(j); g[j].add(i)
    for i in range(n): g.setdefault(i, set())
    exact_rho, _ = brute_force_densest(g)
    greedy_rho, _ = greedy_densest_subgraph(g)
    ratio = exact_rho / greedy_rho if greedy_rho > 0 else float('inf')
    print(f"{n:>4} {exact_rho:>8.3f} {greedy_rho:>8.3f} {ratio:>8.3f}")

## Summary

| Method | Complexity | Guarantee |
|--------|-----------|-----------|
| Greedy (remove min-degree) | O(m) | 2-approximation |
| LP relaxation (Charikar) | Poly | Exact |
| Goldberg max-flow | O(n³ log n) | Exact |

The greedy 2-approximation is near-optimal in practice and runs in linear time.
For web-scale graphs (billions of edges), greedy is the only feasible approach.